# Planetary Hydrothermal Environment Detector
Author: Eriita Jones, Curtin Uni, eriita.jones@curtin.edu.au\
Acknowledgements: ARC Discovery Grant, PI Prof. Katarina Miljkovic

## Accessing CRISM Data for PHYND

### Required data products

PHYND is designed to operate on **CRISM infrared targeted observations**, specifically **IF (I/F) products** reflectance suitable for mineralogical analysis.

You will need:
- A CRISM **IF** image file (`*.img`)
- The corresponding ENVI header file (`*.hdr`)

Other CRISM products (e.g., TRR, MTR, TER) are **not recommended** for this pipeline unless you understand the implications for atmospheric correction and radiometric scaling.  Atmospheric removal via dark pixel subtraction is incorporated in this pipeline.

---

### Where to obtain CRISM data

CRISM data can be accessed through either of the following official NASA archives:

#### Option 1: PDS Geosciences Node (authoritative archive)

- https://pds-geosciences.wustl.edu
- Navigate to:
  - *Mars → Reconnaissance Orbiter → CRISM → Targeted Reduced Data Records (TRDR)*
- Look for products with identifiers containing:
  - `_if` (infrared I/F reflectance)

This option is recommended if you already know the observation ID (e.g., `frt00005e4a`).

#### Option 2: ODE (Orbital Data Explorer; recommended for discovery)

- https://ode.rsl.wustl.edu/mars
- Use the **CRISM search interface** to:
  - Filter by product type: *Targeted*
  - Filter by wavelength range: *Infrared*
  - Identify observations of interest (e.g., impact craters)

ODE is particularly useful for discovering CRISM observations associated with specific geologic features. Links from ODE lead directly to the corresponding PDS data products.


### Product naming conventions

PHYND expects filenames that include:
- `_if` → indicates infrared I/F reflectance
- `.img` → binary image file
- `.hdr` → ENVI header file containing wavelength information



In [61]:
# @title Imports
## Imports

!pip install spectral matplotlib numpy scipy scikit-image --quiet

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.ticker import MultipleLocator
import spectral.io.envi as envi
from scipy.ndimage import uniform_filter, generic_filter, convolve, gaussian_filter1d, binary_dilation
from scipy.signal import find_peaks,savgol_filter
from scipy.spatial import ConvexHull
from skimage import exposure, restoration
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from pathlib import Path
import os
import re
import glob
import pandas as pd

print("Libraries imported successfully.")

Libraries imported successfully.


In [62]:
# @title Read CRISM Metadata

## CRISM ENVI header parsing

def parse_envi_header(hdr_path):
    with open(hdr_path, "r") as f:
        lines = f.readlines()

    metadata = {}
    wavelengths = []
    fwhm = []
    reading_wl = False
    reading_fwhm = False

    def parse_list(line):
        vals = line.replace("{", "").replace("}", "").strip()
        return [v.strip() for v in vals.split(",") if v.strip()]

    for line in lines:
        s = line.strip()
        if not s or s.startswith(";"):
            continue

        keyval = s.split("=", 1)
        if len(keyval) == 2:
            key, val = keyval
            key = key.strip().lower()
            val = val.strip()

            if key == "wavelength" and "{" in val:
                reading_wl = True
            if key == "fwhm" and "{" in val:
                reading_fwhm = True

            if key in ["samples", "lines", "bands", "data type", "header offset"]:
                m = re.search(r"-?\d+", val)
                if m:
                    metadata[key] = int(m.group())

            if key == "interleave":
                metadata[key] = val.lower()

        if reading_wl:
            for v in parse_list(s):
                try:
                    wavelengths.append(float(v))
                except ValueError:
                    pass
            if "}" in s:
                reading_wl = False

        if reading_fwhm:
            for v in parse_list(s):
                try:
                    fwhm.append(float(v))
                except ValueError:
                    pass
            if "}" in s:
                reading_fwhm = False

    return (
        metadata.get("bands"),
        np.array(wavelengths, dtype=np.float32),
        np.array(fwhm, dtype=np.float32),
        metadata
    )



In [63]:
# @title Load CRISM Image
## CRISM image loader

def load_crism(img_path, hdr_path):
    bands, wavelengths, fwhm, meta = parse_envi_header(hdr_path)
    img = envi.open(hdr_path, img_path).load().astype(float)

    data = {
        "image": img,
        "image_corrected": None,
        "wavelengths": wavelengths,
        "fwhm": fwhm,
        "metadata": meta,
        "filename": Path(img_path).name
    }

    print("Loaded:", data["filename"])
    print("Shape:", img.shape)
    print("Wavelength range:", wavelengths.min(), wavelengths.max())

    return data


In [64]:
# @title Radiometric preprocessing
## Dark object subtraction

def dark_object_subtraction(data, window=3, visualize=False):
    img = data["image"]

    summed = img.sum(axis=2)
    min_filtered = uniform_filter(summed, size=window)
    y, x = np.unravel_index(np.argmin(min_filtered), summed.shape)

    dark_spec = img[y, x, :]
    corrected = img - dark_spec
    corrected[corrected < 0] = 0

    data["image_corrected"] = corrected
    data["dark_pixel"] = (y, x)
    data["dark_spectrum"] = dark_spec

    if visualize:
        visualize_dark_pixels(data)

    return data

def visualize_dark_pixels(
    data,
    r_nm=600, g_nm=530, b_nm=430,
    valid_range=(0, 10),
    stretch=(2, 98),
    equalize=True,
    denoise=True,
    gamma=0.6,
    box_size=0,
    crosshair_size=24,
    use_corrected_for_rgb=False,
):

    if "dark_pixel" not in data or "dark_spectrum" not in data:
        raise ValueError(
            "Dark pixel information missing. "
            "Run dark_object_subtraction before visualization."
        )

    wl = np.asarray(data["wavelengths"], dtype=float)
    y, x = data["dark_pixel"]

    cube = (
        data["image_corrected"]
        if (use_corrected_for_rgb and data.get("image_corrected") is not None)
        else data["image"]
    )

    rgb, (r_i, g_i, b_i), (r_w, g_w, b_w) = rgb_composite_crism_style(
        cube,
        wl,
        r_nm=r_nm,
        g_nm=g_nm,
        b_nm=b_nm,
        valid_range=valid_range,
        stretch=stretch,
        equalize=equalize,
        denoise=denoise,
        gamma=gamma,
    )

    fig, ax = plt.subplots(1, 2, figsize=(15, 6))

    # Left panel: RGB image with dark pixel marked
    ax[0].imshow(rgb)
    ax[0].plot(
        x, y,
        marker="+",
        color="red",
        markersize=crosshair_size,
        markeredgewidth=2.5
    )

    if box_size > 0:
        half = box_size / 2
        ax[0].add_patch(
            Rectangle(
                (x - half, y - half),
                box_size,
                box_size,
                linewidth=2,
                edgecolor="red",
                facecolor="none"
            )
        )

    ax[0].set_title(
        "Dark pixel location (RGB composite)\n"
        f"R={r_w:.1f} nm (band {r_i + 1})  "
        f"G={g_w:.1f} nm (band {g_i + 1})  "
        f"B={b_w:.1f} nm (band {b_i + 1})\n"
        f"(row={y}, column={x})",
        fontsize=11,
        fontweight="bold"
    )
    ax[0].axis("off")

    # Right panel: dark pixel spectrum
    dark_spec = np.asarray(data["dark_spectrum"], dtype=float)

    wl_plot = wl.copy()
    if wl_plot.size > 0 and wl_plot.max() < 100:
        wl_plot = wl_plot * 1000.0

    ax[1].plot(wl_plot, dark_spec, color="b", lw=2)
    ax[1].fill_between(wl_plot, 0, dark_spec, alpha=0.25)

    ax[1].set_title("Dark pixel spectrum", fontsize=11, fontweight="bold")
    ax[1].set_xlabel("Wavelength (nm)")
    ax[1].set_ylabel("Signal")
    ax[1].grid(True, alpha=0.3)
    ax[1].set_xlim(wl_plot.min(), wl_plot.max())

    ax[1].text(
        0.02,
        0.98,
        f"mean = {np.nanmean(dark_spec):.4g}\n"
        f"min  = {np.nanmin(dark_spec):.4g}\n"
        f"max  = {np.nanmax(dark_spec):.4g}",
        transform=ax[1].transAxes,
        va="top",
        ha="left",
        bbox=dict(
            boxstyle="round",
            facecolor="white",
            alpha=0.85,
            edgecolor="0.8"
        )
    )

    plt.tight_layout()
    plt.show()


In [65]:
# @title Spectral utilities

def _fill_nan_with_median(arr):
    arr = np.asarray(arr, dtype=float)
    finite = np.isfinite(arr)
    if not np.any(finite):
        return np.zeros_like(arr)
    med = np.median(arr[finite])
    out = arr.copy()
    out[~finite] = med
    return out


def nearest_band(wavelengths, target_nm):
    wavelengths = np.asarray(wavelengths, dtype=float)
    return int(np.argmin(np.abs(wavelengths - target_nm)))


def band_depth(data, center, left, right, use_corrected=True):
    img = data["image_corrected"] if use_corrected else data["image"]
    wl = data["wavelengths"]

    c = nearest_band(wl, center)
    l = nearest_band(wl, left)
    r = nearest_band(wl, right)

    Rc = img[:, :, c]
    Rl = img[:, :, l]
    Rr = img[:, :, r]

    cont = (Rl + Rr) / 2
    cont[cont <= 0] = np.nan

    bd = 1 - Rc / cont
    bd[bd < 0] = 0

    return bd


def band_ratio(data, num_win, den_win, use_corrected=True):
    img = data["image_corrected"] if use_corrected else data["image"]
    wl = data["wavelengths"]

    n_idx = np.where((wl >= num_win[1]) & (wl <= num_win[2]))[0]
    d_idx = np.where((wl >= den_win[1]) & (wl <= den_win[2]))[0]

    if n_idx.size == 0:
        n_idx = [nearest_band(wl, num_win[0])]
    if d_idx.size == 0:
        d_idx = [nearest_band(wl, den_win[0])]

    num = img[:, :, n_idx].mean(axis=2)
    den = img[:, :, d_idx].mean(axis=2)
    den[den == 0] = np.nan

    return num / den


def composite_bd(data, windows, weights, use_corrected=True):
    weights = np.asarray(weights, dtype=float)
    weights /= weights.sum()

    comp = None
    for (c, l, r), w in zip(windows, weights):
        bd = band_depth(data, c, l, r, use_corrected)
        comp = bd * w if comp is None else comp + bd * w

    return comp


def high_low_index(data, high_windows, low_windows, weights=(5, 1), use_corrected=True):
    H = composite_bd(data, high_windows, weights, use_corrected)
    L = composite_bd(data, low_windows, weights, use_corrected)

    strength = H + L
    nd = (H - L) / (strength + 1e-10)

    nd[~np.isfinite(nd)] = np.nan
    return nd, H, L



In [66]:
# @title Visuzalization utilities

def rgb_composite_crism_style(
    cube,
    wavelengths,
    r_nm=600,
    g_nm=530,
    b_nm=440,
    valid_range=(0, 10),
    stretch=(2, 98),
    equalize=True,
    denoise=True,
    denoise_sigma_color=0.05,
    denoise_sigma_spatial=15,
    gamma=0.6,
):

    wl = np.asarray(wavelengths, dtype=float)

    r_i = nearest_band(wl, r_nm)
    g_i = nearest_band(wl, g_nm)
    b_i = nearest_band(wl, b_nm)

    out_bands = []

    for idx in [r_i, g_i, b_i]:
        band = cube[:, :, idx].astype(float)

        lo, hi = valid_range
        band = np.where(
            np.isfinite(band) & (band >= lo) & (band <= hi),
            band,
            np.nan
        )

        band_filled = _fill_nan_with_median(band)

        p_lo, p_hi = np.percentile(band_filled, stretch)
        band_scaled = exposure.rescale_intensity(
            band_filled,
            in_range=(p_lo, p_hi),
            out_range=(0, 1)
        )

        if equalize:
            band_scaled = exposure.equalize_hist(band_scaled)

        if denoise:
            band_scaled = restoration.denoise_bilateral(
                band_scaled,
                sigma_color=denoise_sigma_color,
                sigma_spatial=denoise_sigma_spatial
            )

        p_lo2, p_hi2 = np.percentile(band_scaled, stretch)
        band_out = exposure.rescale_intensity(
            band_scaled,
            in_range=(p_lo2, p_hi2),
            out_range=(0, 1)
        )

        out_bands.append(band_out)

    rgb = np.dstack(out_bands)
    rgb = np.clip(rgb, 0, 1)

    if gamma is not None:
        rgb = rgb ** gamma

    return rgb, (r_i, g_i, b_i), (wl[r_i], wl[g_i], wl[b_i])


def rgb_visualize_masked(
    cube,
    wavelengths,
    moderate_mask,
    weak_mask=None,
    ratio=None,                      # ND index (H-L)/(H+L)
    Hbd_comp=None,                   # High‑T composite
    Lbd_comp=None,                   # Low‑T composite
    show_HGL=False,                  # produce H–G–L RGB
    ratio_cmap="RdBu_r",
    ratio_pct=(2, 98),
    r_nm=600,
    g_nm=530,
    b_nm=440,
    valid_range=(0, 10),
    stretch=(2, 98),
    equalize=True,
    denoise=True,
    denoise_sigma_color=0.05,
    denoise_sigma_spatial=15,
    gamma=0.6,
    overlay_color=(1.0, 0.0, 0.0),        # red = moderate
    weak_overlay_color=(1.0, 0.5, 0.0),   # orange = weak
    alpha=0.6,
    outline=False,
    outline_color=(1.0, 1.0, 1.0),
    outline_lw=0.8,
    expand_radius=3,
    show=True,
    figsize=(10, 10),
    title="Segmented detections",
):

    img = cube["image_corrected"]


    # Build true‑colour RGB
    rgb, band_indices, band_wavelengths = rgb_composite_crism_style(
        img,
        wavelengths,
        r_nm=r_nm,
        g_nm=g_nm,
        b_nm=b_nm,
        valid_range=valid_range,
        stretch=stretch,
        equalize=equalize,
        denoise=denoise,
        denoise_sigma_color=denoise_sigma_color,
        denoise_sigma_spatial=denoise_sigma_spatial,
        gamma=gamma
    )

    # Prepare masks
    mod_mask = np.asarray(moderate_mask).astype(bool)

    if mod_mask.shape != rgb.shape[:2]:
        msg = f"{title}: mask shape mismatch"
        print(msg)
        return None, None, None, None, None, None, None, None, msg

    low, high = valid_range
    full_img = cube["image_corrected"]
    good_pix = np.all(
        np.isfinite(full_img) &
        (full_img >= low) &
        (full_img <= high),
        axis=2
    )

    mod_mask &= good_pix

    if weak_mask is not None:
        weak_mask = np.asarray(weak_mask).astype(bool)
        weak_mask &= good_pix

    if not np.any(mod_mask):
        msg = f"{title}: empty mask"
        print(msg)
        return None, None, None, band_indices, band_wavelengths, (None, None), (None, None), (None, None), msg


    # Expand masks for visualization (radius = 3)
    mod_vis = expand_mask_for_visualization(mod_mask, radius=expand_radius)

    weak_vis = None
    if weak_mask is not None:
        weak_vis = expand_mask_for_visualization(weak_mask, radius=expand_radius)

    vis_mask = mod_vis.copy()
    if weak_vis is not None:
        vis_mask |= weak_vis

    # TRUE‑COLOUR RGB WITH SEGMENTATION
    overlay_rgb = rgb.copy()

    if weak_mask is not None and np.any(weak_mask):
        weak_color = np.array(weak_overlay_color).reshape(1, 1, 3)
        overlay_rgb[weak_mask] = (1 - alpha) * overlay_rgb[weak_mask] + alpha * weak_color

    mod_color = np.array(overlay_color).reshape(1, 1, 3)
    overlay_rgb[mod_mask] = (1 - alpha) * overlay_rgb[mod_mask] + alpha * mod_color

    overlay_rgb = np.clip(overlay_rgb, 0, 1)

    fig_rgb, ax_rgb = None, None
    if show:
        fig_rgb, ax_rgb = plt.subplots(figsize=figsize)
        ax_rgb.imshow(overlay_rgb)
        ax_rgb.set_title(f"{title} – true colour")
        ax_rgb.axis("off")
        plt.show()

    # ND INDEX WITH BLACK‑FILLED SEGMENTATION
    fig_ratio, ax_ratio = None, None
    if show and ratio is not None:
        lo, hi = np.nanpercentile(ratio[np.isfinite(ratio)], ratio_pct)
        ratio_norm = np.clip((ratio - lo) / (hi - lo), 0, 1)
        ratio_rgb = plt.cm.get_cmap(ratio_cmap)(ratio_norm)[..., :3]

        ratio_rgb[vis_mask] *= (1 - 0.7)

        fig_ratio, ax_ratio = plt.subplots(figsize=figsize)
        ax_ratio.imshow(ratio_rgb)
        ax_ratio.set_title("ND Index (H–L)/(H+L), with segmentation (expanded for visualization)")
        ax_ratio.axis("off")
        plt.show()

    # HIGH‑T / DUST / LOW‑T RGB WITH SEGMENTATION
    fig_hgl, ax_hgl = None, None
    if show and show_HGL and (Hbd_comp is not None) and (Lbd_comp is not None):

        wl = np.asarray(wavelengths)
        g_idx = np.argmin(np.abs(wl - 750))
        dust = img[..., g_idx]

        pseudo_cube = np.dstack([Hbd_comp, dust, Lbd_comp])
        pseudo_wl = np.array([2300, 750, 1900], dtype=float)

        rgb_HGL, _, _ = rgb_composite_crism_style(
            pseudo_cube,
            pseudo_wl,
            r_nm=2300,
            g_nm=750,
            b_nm=1900,
            stretch=stretch,
            gamma=gamma,
            equalize=False,
            denoise=False
        )

        rgb_HGL[vis_mask] *= (1 - 0.7)

        fig_hgl, ax_hgl = plt.subplots(figsize=figsize)
        ax_hgl.imshow(rgb_HGL)
        ax_hgl.set_title("High‑T (Red), Dust 750nm (Green), Low‑T (Blue), with segmentation (expanded for visualization)")
        ax_hgl.axis("off")
        plt.show()

    return (
        rgb,
        overlay_rgb,
        band_indices,
        band_wavelengths,
        (fig_rgb, ax_rgb),
        (fig_ratio, ax_ratio),
        (fig_hgl, ax_hgl),
        None
    )


def expand_mask_for_visualization(mask, radius=2):

    mask = np.asarray(mask).astype(bool)
    structure = np.ones((2*radius + 1, 2*radius + 1), dtype=bool)
    return binary_dilation(mask, structure=structure)


def visualize_high_low_index(index, Hbd, Lbd,
                              index_range=(-1, 1),
                              cmap="RdBu_r"):

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # High-T composite
    h2, h98 = np.nanpercentile(Hbd, [2, 98])
    im0 = axes[0].imshow(Hbd, cmap="viridis", vmin=h2, vmax=h98)
    axes[0].set_title("High-T composite band depth (H)")
    axes[0].axis("off")
    plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    # Low-T composite
    l2, l98 = np.nanpercentile(Lbd, [2, 98])
    im1 = axes[1].imshow(Lbd, cmap="viridis", vmin=l2, vmax=l98)
    axes[1].set_title("Low-T composite band depth (L)")
    axes[1].axis("off")
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    # Normalized difference index
    im2 = axes[2].imshow(index, cmap=cmap,
                          vmin=index_range[0],
                          vmax=index_range[1])
    axes[2].set_title("Normalized difference index (H − L)/(H + L)")
    axes[2].axis("off")
    plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

In [67]:
# @title Segmentation
## Segmentation with local background comparison

def nanpercentile_filter2d(
    img,
    percentile,
    win=51,
    min_valid=0.5,
    exclude_center=True
):
    img = np.asarray(img, dtype=np.float32)

    # Valid data mask
    valid = np.isfinite(img).astype(np.float32)

    kernel = np.ones((win, win), dtype=np.float32)

    # Count valid pixels in each window
    local_cnt = convolve(valid, kernel, mode="constant", cval=0.0)

    if exclude_center:
        local_cnt -= valid

    required = min_valid * (win * win - (1 if exclude_center else 0))

    k = win * win
    center_idx = k // 2


    def _pct_func(values):
        if exclude_center:
            values = np.delete(values, center_idx)

        values = values[np.isfinite(values)]
        if values.size < required:
            return np.nan

        return np.nanpercentile(values, percentile)

    local_pct = generic_filter(
        img,
        _pct_func,
        size=win,
        mode="constant",
        cval=np.nan
    )

    local_pct[local_cnt < required] = np.nan

    return local_pct, local_cnt


def score3(x, weak, moderate):
    s = np.zeros_like(x, dtype=np.uint8)
    s[x >= weak] = 1
    s[x >= moderate] = 2
    return s


def quality_index_local_background(
    bd1, bd2, ratio,
    # absolute thresholds
    bd1_weak=0.001, bd1_mod=0.015,
    bd2_weak=0.005, bd2_mod=0.010,
    ratio_weak=1.001, ratio_mod=1.05,

    # local percentile thresholds
    bd1_p_weak=75, bd1_p_mod=90,
    bd2_p_weak=75, bd2_p_mod=90,
    ratio_p_weak=75, ratio_p_mod=90,

    win=51, min_valid=0.6, exclude_center=True,
    gate=True
):
    bd1   = np.asarray(bd1, dtype=np.float32)
    bd2   = np.asarray(bd2, dtype=np.float32)
    ratio = np.asarray(ratio, dtype=np.float32)

    # Require finite values in all diagnostic inputs
    valid = np.isfinite(bd1) & np.isfinite(bd2) & np.isfinite(ratio)


    # Absolute scores
    s1_abs = score3(bd1,   bd1_weak,  bd1_mod) * valid
    s2_abs = score3(bd2,   bd2_weak,  bd2_mod) * valid
    sr_abs = score3(ratio, ratio_weak, ratio_mod) * valid


    # The dominance metric (ratio or ND index) is used as corroborating evidence
    # rather than a primary detection criterion.
    candidate = (s1_abs >= 1) | (s2_abs >= 1)

    # Local percentile maps (per-pixel thresholds derived from neighborhood)
    bd1_cand = np.where(candidate & valid, bd1, np.nan)
    bd2_cand = np.where(candidate & valid, bd2, np.nan)
    rat_cand = np.where(candidate & valid, ratio, np.nan)


    bd1_w_pct, bd1_cnt = nanpercentile_filter2d(
        bd1_cand, bd1_p_weak, win=win, min_valid=min_valid, exclude_center=exclude_center
    )
    bd1_m_pct, _ = nanpercentile_filter2d(
        bd1_cand, bd1_p_mod, win=win, min_valid=min_valid, exclude_center=exclude_center
    )

    bd2_w_pct, bd2_cnt = nanpercentile_filter2d(
        bd2_cand, bd2_p_weak, win=win, min_valid=min_valid, exclude_center=exclude_center
    )
    bd2_m_pct, _ = nanpercentile_filter2d(
        bd2_cand, bd2_p_mod, win=win, min_valid=min_valid, exclude_center=exclude_center
    )

    rat_w_pct, rat_cnt = nanpercentile_filter2d(
        rat_cand, ratio_p_weak, win=win, min_valid=min_valid, exclude_center=exclude_center
    )
    rat_m_pct, _ = nanpercentile_filter2d(
        rat_cand, ratio_p_mod, win=win, min_valid=min_valid, exclude_center=exclude_center
    )

    # Combine absolute thresholds + percentile thresholds
    # Use the stricter (higher) threshold at each pixel:
    bd1_w_eff = np.maximum(bd1_weak, bd1_w_pct)
    bd1_m_eff = np.maximum(bd1_mod,  bd1_m_pct)

    bd2_w_eff = np.maximum(bd2_weak, bd2_w_pct)
    bd2_m_eff = np.maximum(bd2_mod,  bd2_m_pct)

    rat_w_eff = np.maximum(ratio_weak, rat_w_pct)
    rat_m_eff = np.maximum(ratio_mod,  rat_m_pct)

    # percentile/distinctness scores
    s1_pct = score3(bd1,   bd1_w_eff, bd1_m_eff)
    s2_pct = score3(bd2,   bd2_w_eff, bd2_m_eff)
    sr_pct = score3(ratio, rat_w_eff, rat_m_eff)

    # Enforce BOTH absolute + percentile
    s1 = np.minimum(s1_abs, s1_pct)
    s2 = np.minimum(s2_abs, s2_pct)
    sr = np.minimum(sr_abs, sr_pct)
    sr_eff = np.maximum(sr, sr_abs)


    # Core high‑T assemblages: strong absolute high‑T signal and clear dominance
    # over low‑T diagnostics. Intended to capture interiors of coherent units.
    core_highT = (
        valid &
        (s1_abs >= 2) &
        (bd1 >= bd1_m_eff) &
        (ratio >= rat_m_eff)
    )

    # combine into Q
    if gate:
        detected = (
        (
            ((s1 >= 1) | (s2 >= 1)) & (sr >= 1)
        ) |
        core_highT
        )

        detected = (
        (
            ((s1 >= 1) )
        ) |
        core_highT
        )


        Q = np.zeros_like(s1, dtype=np.uint8)
        Q[detected] = (s1[detected] + s2[detected] + sr_eff[detected]).astype(np.uint8)
    else:
        Q = (s1 + s2 + sr_eff).astype(np.uint8)

    # Return diagnostics
    comps = {
            # final scores
            "score_bd1": s1,
            "score_bd2": s2,
            "score_ratio": sr_eff,

            # absolute-only
            "score_bd1_abs": s1_abs,
            "score_bd2_abs": s2_abs,
            "score_ratio_abs": sr_abs,

            # percentile-only
            "score_bd1_pct": s1_pct,
            "score_bd2_pct": s2_pct,
            "score_ratio_pct": sr_pct,

            # local percentile thresholds
            "local_pct_bd1_weak": bd1_w_pct,
            "local_pct_bd1_mod":  bd1_m_pct,
            "local_pct_bd2_weak": bd2_w_pct,
            "local_pct_bd2_mod":  bd2_m_pct,
            "local_pct_ratio_weak": rat_w_pct,
            "local_pct_ratio_mod":  rat_m_pct,

            # effective thresholds
            "effective_bd1_weak": bd1_w_eff,
            "effective_bd1_mod":  bd1_m_eff,
            "effective_bd2_weak": bd2_w_eff,
            "effective_bd2_mod":  bd2_m_eff,
            "effective_ratio_weak": rat_w_eff,
            "effective_ratio_mod":  rat_m_eff,

            # neighborhood support
            "local_valid_count_bd1": bd1_cnt,
            "local_valid_count_bd2": bd2_cnt,
            "local_valid_count_ratio": rat_cnt,
        }

    return Q, comps


def analyze_segmented_spectrum_features(
    data,
    mask,
    use_percentiles=2,
    valid_range=(-10, 10),

    smooth_sigma=1,

    min_band_depth=0.002,
    min_prominence=0.002,
    peak_width=2,

    wl_min_um=1.4,
    wl_max_um=2.6,

    plot=True,
    title="Mean spectrum of segmented pixels"
):

    img = data["image_corrected"]
    wl_nm = np.asarray(data["wavelengths"], dtype=float)
    wl_um = wl_nm / 1000.0
    mask = np.asarray(mask).astype(bool)

    # Validity filtering
    low, high = valid_range
    good_pix = np.all(
        np.isfinite(img) &
        (img > low) &
        (img < high),
        axis=2
    )
    mask = mask & good_pix

    if not np.any(mask):
        print("Mask empty after validity filtering.")
        return None, None

    # Extract spectra and compute mean
    specs = img[mask, :]
    mean_spec = np.nanmean(specs, axis=0)

    if use_percentiles is not None:
        lo = np.nanpercentile(specs, use_percentiles, axis=0)
        hi = np.nanpercentile(specs, 100 - use_percentiles, axis=0)
    else:
        lo = hi = None

    # Savitzky–Golay smoothing (preserves absorption edges)
    spec_smooth = savgol_filter(
        mean_spec,
        window_length=11,  # must be odd; tune to CRISM sampling
        polyorder=3
    )

    # First derivative with respect to wavelength (nm)
    d_spec = np.gradient(spec_smooth, wl_nm)

    # Continuum removal (convex hull)
    pts = np.column_stack((wl_nm, spec_smooth))
    hull = ConvexHull(pts)
    hull_idx = np.unique(hull.vertices)
    hull_idx = hull_idx[np.argsort(wl_nm[hull_idx])]

    continuum = np.interp(
        wl_nm,
        wl_nm[hull_idx],
        spec_smooth[hull_idx]
    )

    cr = 1.0 - spec_smooth / continuum

    # Restrict to 1.4–2.6 µm for peak detection
    feat_mask = (wl_um >= wl_min_um) & (wl_um <= wl_max_um)
    wl_sub = wl_nm[feat_mask]
    d_sub = d_spec[feat_mask]

    rows = []

    # Peak-based detection (narrow features)
    peaks, props = find_peaks(
        -d_sub,
        prominence=min_prominence,
        width=peak_width
    )

    for i, p in enumerate(peaks):

        slope_mag = -d_sub[p]  # positive magnitude

        if slope_mag < min_band_depth:
            continue

        wl_peak_um = wl_sub[p] / 1000.0

        # Skip basin region (handled separately)
        if 2.30 <= wl_peak_um <= 2.40:
            continue

        if 2.20 <= wl_peak_um <= 2.25:
            interp = (
                "Al-OH | narrow peak | "
                "illite, prehnite, Al-rich smectites"
            )
        elif 1.38 <= wl_peak_um <= 1.42:
            interp = (
                "OH overtone | narrow peak | "
                "phyllosilicates (general)"
            )
        elif 1.85 <= wl_peak_um <= 1.95:
            interp = (
                "H2O combination | narrow peak | "
                "smectites, hydrated alteration minerals"
            )
        else:
            interp = (
                "Unclassified absorption | narrow peak | "
                "ambiguous or minor phase"
            )

        rows.append({
          "Edge center (µm)": wl_peak_um,
          "Slope magnitude": slope_mag,
          "Prominence": props["prominences"][i],
          "Detection type": "Derivative edge",
          "Interpretation": interp
        })

    # Basin-based detection for 2.30–2.40 µm Fe/Mg-OH
    # basin window
    basin_mask = (wl_um >= 2.30) & (wl_um <= 2.40)

    wl_basin = wl_nm[basin_mask]
    spec_basin = spec_smooth[basin_mask]

    # local linear continuum between basin edges
    cont_basin = np.interp(
        wl_basin,
        [wl_basin[0], wl_basin[-1]],
        [spec_basin[0], spec_basin[-1]]
    )

    # local continuum-removed basin depth
    cr_basin_local = 1.0 - spec_basin / cont_basin
    basin_depth = np.nanmax(cr_basin_local)

    if basin_depth >= min_band_depth:
        idx = np.nanargmax(cr_basin_local)
        wl_center_um = wl_basin[idx] / 1000.0

        rows.append({
            "Absoption edge (µm)": wl_center_um,
            "Band depth": basin_depth,

        })

    feature_table = pd.DataFrame(rows)

    if plot:
        from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

        # --- Figure and main axes ---
        fig, ax = plt.subplots(figsize=(12, 5))

        # Mean spectrum
        ax.plot(wl_nm, mean_spec, color="k", lw=2, label="Mean spectrum")

        # Envelope
        if lo is not None:
            ax.fill_between(
                wl_nm, lo, hi,
                color="lightgray", alpha=0.3,
                label=f"{use_percentiles}–{100-use_percentiles}% envelope"
            )

        # Diagnostic wavelengths (nm)
        diag_wls = [1900, 2100, 2300, 2360]  # adjust if needed

        # Green lines on MAIN axes
        for wl in diag_wls:
            ax.axvline(wl, color="tab:green", lw=1.5, ls="--", alpha=0.9)

        # Labels and title (set ONCE)
        ax.set_xlabel("Wavelength (nm)")
        ax.set_ylabel("Reflectance (I/F)")
        ax.set_title(
            f"{title}\nN pixels = {specs.shape[0]}",
            fontsize=12,
            pad=12
        )

        ax.grid(True, alpha=0.3)
        ax.legend(loc="upper left")

        # --- Zoomed inset (UPPER RIGHT) ---
        x1, x2 = 1800, 2600
        zoom_mask = (wl_nm >= x1) & (wl_nm <= x2)

        axins = inset_axes(
            ax,
            width="38%",
            height="38%",
            loc="upper right",
            borderpad=1.5
        )

        axins.plot(wl_nm[zoom_mask], mean_spec[zoom_mask], color="k", lw=2)

        if lo is not None:
            axins.fill_between(
                wl_nm[zoom_mask],
                lo[zoom_mask],
                hi[zoom_mask],
                color="lightgray",
                alpha=0.4
            )

        # Green lines on INSET
        for wl in diag_wls:
            if x1 <= wl <= x2:
                axins.axvline(wl, color="tab:green", lw=1.2, ls="--", alpha=0.9)

        # Tight y-scaling for inset
        ymin = np.nanmin(mean_spec[zoom_mask])
        ymax = np.nanmax(mean_spec[zoom_mask])
        pad = 0.06 * (ymax - ymin)
        axins.set_ylim(ymin - pad, ymax + pad)

        # Major ticks every 100 nm (labeled)
        axins.xaxis.set_major_locator(MultipleLocator(100))

        # Minor ticks every 20 nm (tick marks only)
        axins.xaxis.set_minor_locator(MultipleLocator(20))

        # Tick appearance
        axins.tick_params(axis="x", which="major", length=6, width=1.0, labelsize=8)
        axins.tick_params(axis="x", which="minor", length=3, width=0.8)
        axins.set_yticks([])
        axins.set_title("Zoom: 1.9–2.6 µm", fontsize=9)
        axins.grid(True, alpha=0.3)

        # Optional connector box
        mark_inset(ax, axins, loc1=2, loc2=4, fc="none", ec="0.4", lw=1.0)

        plt.tight_layout()
        plt.show()


    return mean_spec, feature_table


In [72]:
# =========================
# USER PARAMETERS
# =========================
DO_DARK = True
ALL_IMAGES = False
IDX = 1 # Only used if ALL_IMAGES = True
DRIVE_PATH = "/content/drive/MyDrive/CRISM/"
IMAGE_ID = "frt00009786_07_if165j_mtr3" # Only used if ALL_IMAGES = False

# =========================
# LOAD DATA
# =========================
from google.colab import drive
drive.mount('/content/drive')

if not(ALL_IMAGES):
  img_path = os.path.join(DRIVE_PATH,IMAGE_ID+".img")
  hdr_path = os.path.join(DRIVE_PATH,IMAGE_ID+".hdr")

  if "_if" not in os.path.basename(img_path):
    raise ValueError(f"Selected file is not an IF product: {img_path}")

  data = load_crism(img_path, hdr_path)

  if DO_DARK:
    data = dark_object_subtraction(data)


if ALL_IMAGES:
  # Construct the pattern and find files
  img_files = glob.glob(os.path.join(DRIVE_PATH, f"*.img"))
  img_path = img_files[IDX]

  if "_if" not in os.path.basename(img_path):
    raise ValueError(f"Selected file is not an IF product: {img_path}")

  base, _ = os.path.splitext(img_path)
  hdr_path = base + ".hdr"

  if not os.path.exists(hdr_path):
    print(f'Error: HDR file {base} could not be found')
  print(img_path,hdr_path)
  data = load_crism(img_path, hdr_path)

  if DO_DARK:
    data = dark_object_subtraction(data, visualize = True)


# =========================
# ANALYSIS
# =========================
# NOTE:
# Composite band-depth images are computed internally by high_low_index
# as the mean of multiple diagnostic band-depth maps for each assemblage.
# These composites are used consistently for visualization, indexing,
# and segmentation.


# Define diagnostic bands (nm)
# Low-T features: hydration + Al-OH
L1 = [1925, 1900, 1950]   # H2O combination band
L2 = [2387, 2375, 2400]   # Al-OH / mixed low-T OH
L3 = [1425, 1400, 1450]   # OH overtone
bd_low_windows= [L1,L2,L3]

# High-T features: broad Fe/Mg-OH
H1 = [2350, 2300, 2400]   # broad Fe/Mg-OH basin (chlorite/epidote)
H2 = [2587, 2575, 2600]   # long-wavelength OH
H3 = [2537, 2525, 2550]   # shoulder region
bd_high_windows = [H1,H2,H3]

# Compute composite high-T and low-T band-depth images and
# a normalized difference index expressing relative dominance
# of high- versus low-temperature assemblages.
index, Hbd_comp, Lbd_comp = high_low_index(data, bd_high_windows, bd_low_windows, use_corrected=DO_DARK)
visualize_high_low_index(index, Hbd_comp, Lbd_comp)



# =========================
# SEGMENTATION
# =========================
# Segmentation logic:
# Pixels must exceed both absolute and background-relative thresholds.
# Assemblage type (high-T vs low-T) is determined by relative dominance
# of Fe/Mg-OH versus hydration/Al-OH diagnostics. Segmentation is performed on composite band-depth maps that
# average multiple diagnostic absorptions for each assemblage type.
# The ND index (range -1 to +1) is used as a supporting metric rather than a primary detection criterion.


# Convention:
# bd1 = high-temperature composite diagnostic
# bd2 = low-temperature composite diagnostic

Q, comps = quality_index_local_background(
    Hbd_comp,   # bd1: High-T
    Lbd_comp,    # bd2: Low-T
    ratio=index,

    # absolute thresholds
    bd1_weak=0.05,   # high-T weak
    bd1_mod=0.07,     # high-T moderate

    bd2_weak=0.06,   # low-T weak
    bd2_mod=0.08,     # low-T moderate

    # background-relative thresholds
    bd1_p_weak=50,
    bd1_p_mod=85,

    bd2_p_weak=50,
    bd2_p_mod=85,

    ratio_weak=0.01,
    ratio_mod=0.3,

    win=50,
    min_valid=0.6,
    exclude_center=True,
    gate=True
)

print("Final detections:", np.count_nonzero(Q))

# High-temperature assemblages:
# Dominance criterion: require high‑T diagnostics to dominate over low‑T
# diagnostics based on the normalized difference index. This allows interiors
# of spatially coherent high‑T assemblages to be retained even when local
# percentile contrast is low.

#Moderate‑to‑strong high‑T signal, regardless of low‑T.
high_highlight_mask = (
    (Q > 0) &
    (comps["score_bd1"] >= 2) &
    (index >= 0.1)
)

high_weak_mask = (
    (Q > 0) &
    (comps["score_bd1"] >= 1) &
    (comps["score_bd1"] >= comps["score_bd2"])
)

#Moderate‑to‑strong high‑T and dominance over low‑T.
#high_highlight_mask = (
#    (Q > 0) &
#    (comps["score_bd1"] >= 2) &
#    (comps["score_bd1"] >= comps["score_bd2"])
#    (index >= 0.1)
#)


# Low-temperature assemblages:
# require dominance of low-T signal
#low_highlight_mask = (
#    (Q > 0) &
#    (comps["score_bd2"] > comps["score_bd1"])
#)

# Visualize segmented pixels on natural look image.
highT_mask_vis = expand_mask_for_visualization(
    high_highlight_mask,
    radius=2
)
highT_weakmask_vis = expand_mask_for_visualization(
    high_weak_mask,
    radius=2
)

rgb, overlay_rgb, band_idx, band_wl, _, _,_,err = rgb_visualize_masked(
    data, data["wavelengths"],
    moderate_mask=highT_mask_vis,
    ratio = index,
    Hbd_comp = Hbd_comp,
    Lbd_comp = Lbd_comp,
    show_HGL = True,   # Turn on High-G-Low composite
    alpha=0.6,      # Transparency
    outline=True,
    gamma=0.6,
    title="CRISM RGB with highT-assemblages detected (expanded for visualization)"
)

# =========================
# INTERPRETATION
# =========================

mean_spec, feature_table = analyze_segmented_spectrum_features(
    data,
    high_highlight_mask,
    use_percentiles=2, # percentile data clipping
    valid_range=(-10, 10), # max-min data thresholds
    smooth_sigma=1, #spectral smoothing
    min_band_depth=0.001, #threshold for feature identificaiton
    min_prominence=0.001, #threshold for feature identificaiton
    peak_width=2,
    wl_min_um=1.28, #analysis window
    wl_max_um=2.58, #analysis window
    title="High-T assemblages: mean spectrum and automated absoprtion feature analysis"
)








Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


EOFError: read() didn't return enough bytes